# Chapter 22 — Causal Inference Introduction
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Distinguish correlation from causation rigorously
- Understand confounding, selection bias, and collider bias
- Apply difference-in-differences and propensity score matching

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — The Fundamental Problem of Causal Inference

The **Potential Outcomes Framework** (Rubin):

For unit $i$:
- $Y_i(1)$ = outcome if treated
- $Y_i(0)$ = outcome if not treated
- **Individual Treatment Effect (ITE):** $\tau_i = Y_i(1) - Y_i(0)$
- **Average Treatment Effect (ATE):** $\tau = E[Y(1) - Y(0)]$

**The fundamental problem:** We can only observe one potential outcome —
the **counterfactual** is never seen.

Random assignment ensures:
$$E[Y(1)|T=1] = E[Y(1)|T=0] \implies \hat\tau = \bar Y_{treated} - \bar Y_{control}$$

In [ ]:
# Simulate: confounding bias without randomisation
n = 1000
# True treatment effect = 2.0
# Confounder: age affects both treatment assignment and outcome
age = rng.normal(40, 10, n)
# Treatment: older people more likely to get treatment
prob_treated = 1 / (1 + np.exp(-(age - 40) / 5))
treated = rng.binomial(1, prob_treated)
# Outcome: treatment adds 2, but age also increases outcome
outcome = 2*treated + 0.3*age + rng.normal(0, 1, n)

# Naive estimate (biased)
ate_naive = outcome[treated==1].mean() - outcome[treated==0].mean()
print(f"True ATE: 2.0")
print(f"Naive ATE (ignoring confounding): {ate_naive:.3f}")

# Correct: control for age with regression
import statsmodels.api as sm
X_reg = sm.add_constant(np.c_[treated, age])
model = sm.OLS(outcome, X_reg).fit()
ate_adjusted = model.params[1]
print(f"Regression-adjusted ATE: {ate_adjusted:.3f}")

## 2. Math Walkthrough — Difference-in-Differences

DiD identifies causal effect from panel data under **parallel trends**:
$$\hat\tau_{DiD} = (\bar Y^{post}_{treat} - \bar Y^{pre}_{treat}) - (\bar Y^{post}_{control} - \bar Y^{pre}_{control})$$

In [ ]:
# Simulate DiD: new checkout feature rolled out to Group B
n_users = 500
# Pre-period
revenue_pre_A = rng.normal(100, 20, n_users)  # Control
revenue_pre_B = rng.normal(95, 20, n_users)   # Treatment (slightly lower pre)

# Post-period: treatment effect = +10
revenue_post_A = rng.normal(103, 20, n_users)  # Control: small time trend
revenue_post_B = rng.normal(108, 20, n_users)  # Treatment: time trend + effect

did = ((revenue_post_B.mean() - revenue_pre_B.mean()) -
       (revenue_post_A.mean() - revenue_pre_A.mean()))
print(f"True treatment effect: +10")
print(f"DiD estimate: {did:.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
periods = ["Pre", "Post"]
ax.plot(periods, [revenue_pre_A.mean(), revenue_post_A.mean()], "b-o", label="Control", lw=2)
ax.plot(periods, [revenue_pre_B.mean(), revenue_post_B.mean()], "r-o", label="Treatment", lw=2)
cf_B = revenue_pre_B.mean() + (revenue_post_A.mean() - revenue_pre_A.mean())
ax.plot(periods, [revenue_pre_B.mean(), cf_B], "r--", label="Counterfactual (no treatment)", lw=2)
ax.annotate(f"DiD={did:.1f}", xy=("Post", (revenue_post_B.mean()+cf_B)/2),
            fontsize=11, color="darkred")
ax.set_ylabel("Revenue"); ax.legend(); ax.set_title("Difference-in-Differences")
plt.tight_layout(); plt.show()

## 3–6. Propensity Score Matching, Colliders, and Practice

In [ ]:
# Propensity Score Matching (PSM)
df = pd.DataFrame({"age": age, "treated": treated, "outcome": outcome})

# Fit propensity score model
lr = LogisticRegression()
lr.fit(df[["age"]], df["treated"])
df["pscore"] = lr.predict_proba(df[["age"]])[:, 1]

# Greedy 1:1 nearest-neighbor matching
treated_df  = df[df.treated == 1].copy()
control_df  = df[df.treated == 0].copy()
matched_pairs = []
used_controls = set()
for _, t_row in treated_df.iterrows():
    diffs = abs(control_df[~control_df.index.isin(used_controls)]["pscore"] - t_row["pscore"])
    best  = diffs.idxmin()
    used_controls.add(best)
    matched_pairs.append((t_row["outcome"], control_df.loc[best, "outcome"]))

matched = np.array(matched_pairs)
ate_psm = (matched[:, 0] - matched[:, 1]).mean()
print(f"PSM ATE: {ate_psm:.3f} (true=2.0)")

In [ ]:
# Practice: confounders in model evaluation
# Students who take an ML course (treated) tend to have higher GPA already
n = 300
gpa_prior = rng.normal(3.2, 0.5, n)
prob_takes_course = 1/(1+np.exp(-(gpa_prior - 3.2)/0.3))
takes_course = rng.binomial(1, prob_takes_course)
# Course adds 0.1 to GPA
gpa_post = gpa_prior + 0.1*takes_course + rng.normal(0, 0.2, n)

naive_effect = gpa_post[takes_course==1].mean() - gpa_post[takes_course==0].mean()

import statsmodels.api as sm
X_c = sm.add_constant(np.c_[takes_course, gpa_prior])
m = sm.OLS(gpa_post, X_c).fit()
adjusted_effect = m.params[1]

print(f"True course effect: 0.1")
print(f"Naive estimate: {naive_effect:.3f}  ← inflated by confounding")
print(f"Regression-adjusted: {adjusted_effect:.3f}  ← closer to truth")

## 🎯 Track 3 Complete! 🏆

**You've mastered:**
- ✅ Distributions in ML and log-likelihood
- ✅ MLE: analytical and numerical estimation
- ✅ Bias-variance tradeoff and regularisation
- ✅ Bayesian vs Frequentist perspectives
- ✅ Hypothesis testing for model evaluation
- ✅ Bootstrap and Jackknife resampling
- ✅ Correlation, covariance, and feature selection
- ✅ Information theory: entropy, KL divergence, cross-entropy
- ✅ Power analysis and experiment design
- ✅ Causal inference: DiD, PSM, confounders

**Ready for Tier 3?** → [Chapter 23 — Markov Chains]